# Week 1 Listing Remark Exploration

This notebook takes a first pass through the artifacts created for Week 1.

In [1]:
from collections import Counter
import json
import re

import pandas as pd

---

## 1. Artifacts Loading

Let's start with loading the three assets created this week:

- a sample of listing records
- the cleaned real estate taxonomy
- labeled user queries for later parser and intent work

In [4]:
listings = pd.read_csv('../data/processed/listing_sample.csv')

with open('../data/processed/taxonomy.json') as f:
    taxonomy = json.load(f)

with open('../data/processed/sample_queries.json') as f:
    queries = json.load(f)

terms = pd.DataFrame(taxonomy['terms'])
queries_df = pd.DataFrame(queries)

---

## 2. Listing Sample

### 2.1 Shape and Fields

Before looking at language, check the shape of the sample and the fields available.

In [5]:
listings.shape

(1000, 7)

In [6]:
listings.head(3)

,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1159550963,29667 Fortitude Drive,Menifee,5.0,4.0,752490,Move in just in time for summer! Our popular ...
1,1155343150,11379 Reidy Canyon,Escondido,8.0,8.0,2825000,Rare multi-generational estate offering three ...
2,1158531238,2689 E Towhee,Ontario,4.0,3.0,855000,"Welcome to 2689 E Towhee St., an exquisite sin..."


The key text field is `remarks`. The remaining fields give useful context for filters and later query parsing.

### 2.2 Basic Field Profile

A quick field profile including type, missingness, and whether the values look usable.

In [7]:
field_profile = pd.DataFrame({
    'dtype': listings.dtypes,
    'missing': listings.isna().sum(),
    'missing_pct': listings.isna().mean().round(3)
})

field_profile

,dtype,missing,missing_pct
L_ListingID,int64,0,0.000
L_Address,object,0,0.000
L_City,object,2,0.002
beds,float64,2,0.002
baths,float64,0,0.000
price,int64,0,0.000
remarks,object,0,0.000


In [8]:
listings[['beds', 'baths', 'price']].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
beds,998.0,3.39,1.98,0.0,2.0,3.0,4.0,37.0
baths,1000.0,2.90,2.32,0.0,2.0,3.0,3.0,46.0
price,1000.0,1804752.23,4409525.95,55000.0,572250.0,887500.0,1549000.0,65000000.0


The structured fields are useful for later parser work:

- `beds`, `baths`, and `price` can become direct filters.
- `city` can support location search.
- `remarks` is where most amenity, condition, and lifestyle signals live.

### 2.3 Geography and Price

The city mix gives a rough view of where the text is coming from. Price buckets help set expectations for the kind of inventory represented in the remarks.

In [9]:
listings['L_City'].value_counts().head(15)

L_City
Los Angeles       62
San Diego         50
San Jose          25
Malibu            14
La Quinta         13
Irvine            13
Indio             12
Victorville       12
Palmdale          11
Valencia          10
Palm Desert       10
Escondido          9
Lake Arrowhead     9
Hemet              8
Long Beach         8
Name: count, dtype: int64

In [10]:
price_bins = pd.cut(
    listings['price'],
    bins=[0, 500000, 750000, 1000000, 1500000, 2500000, 10000000],
    labels=['<500k', '500k-750k', '750k-1m', '1m-1.5m', '1.5m-2.5m', '2.5m+']
)

price_bins.value_counts(sort=False)

price
<500k        196
500k-750k    222
750k-1m      161
1m-1.5m      169
1.5m-2.5m    133
2.5m+         97
Name: count, dtype: int64

### 2.4 Remark Length and Text Density

Listing remarks vary from short descriptions to long marketing copy. Length is a useful proxy for how much NLP signal each listing can provide.

In [11]:
listings['remark_chars'] = listings['remarks'].str.len()
listings['remark_words'] = listings['remarks'].str.split().str.len()

listings[['remark_chars', 'remark_words']].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
remark_chars,1000.0,1277.9,574.0,83.0,874.2,1227.5,1587.0,3963.0
remark_words,1000.0,192.3,84.1,12.0,135.0,184.0,238.0,598.0


In [12]:
listings[
    ['L_ListingID', 'L_City', 'price', 'remark_words', 'remarks']
].sort_values('remark_words', ascending=False).head(5)

,L_ListingID,L_City,price,remark_words,remarks
11,1158609560,Valley Village,1549000,598,Begin each morning in your own reimagined sanc...
395,1150769431,Imperial Beach,1800000,515,New construction end unit at 807 Seacoast that...
717,1151866654,Malibu,14495000,487,"Enter through a grand metal, glass, and wood f..."
126,1172025846,San Marcos,2860000,477,Stunning luxury residence located in The Summi...
200,1118391819,Santa Barbara,14995000,474,A rare midcentury modern architectural offerin...


Longer remarks are especially useful for taxonomy work because they include more features, context, and lifestyle language. Shorter remarks still matter, but they are less likely to cover multiple categories.

### 2.5 Common Listing Language

The first language pass looks for repeated terms and phrases. This is the fastest way to see what agents emphasize before any modeling begins.

In [13]:
stop_words = {
    'the', 'and', 'for', 'with', 'this', 'that', 'you', 'your', 'are', 'from', 'has', 'have',
    'home', 'property', 'offers', 'features', 'including', 'into', 'its', 'all', 'new', 'will'
}

token_re = re.compile(r'[a-z][a-z\'-]+')

def tokens(text):
    return [t for t in token_re.findall(text.lower()) if len(t) > 2 and t not in stop_words]

words = [token for text in listings['remarks'] for token in tokens(text)]

pd.DataFrame(Counter(words).most_common(25), columns=['term', 'count'])

,term,count
0,living,1778
1,room,1260
2,space,1019
3,kitchen,957
4,private,814
5,bedroom,770
6,dining,722
7,spacious,682
8,area,668
9,bedrooms,627


In [14]:
bigrams = []

for text in listings['remarks']:
    row_tokens = tokens(text)
    bigrams.extend(zip(row_tokens, row_tokens[1:]))

pd.DataFrame(
    [(' '.join(pair), count) for pair, count in Counter(bigrams).most_common(25)],
    columns=['phrase', 'count']
)

,phrase,count
0,primary suite,339
1,natural light,313
2,living room,305
3,floor plan,269
4,living space,256
5,car garage,183
6,bedroom bath,180
7,stainless steel,178
8,walk-in closet,178
9,pool spa,169


The top words and bigrams give a first read on the domain vocabulary. Some phrases are direct taxonomy candidates, while others are generic marketing language that should be filtered out during cleaning.

---

## 3 Taxonomy

### 3.1 Taxonomy Shape

The taxonomy should cover a broad set of listing concepts. Category counts show whether the first version is balanced enough for downstream extraction work.

In [15]:
terms['category'].value_counts().reindex(taxonomy['categories'])

category
property_type             22
room                      50
amenity                   30
interior_feature          63
exterior_feature          36
location                  44
condition                 31
transaction_or_listing    28
Name: count, dtype: int64

In [16]:
terms.groupby('category').head(5)[['category', 'term', 'frequency', 'ngram_type', 'review_status']]

,category,term,frequency,ngram_type,review_status
0,property_type,condo,112,unigram,sample_supported_seed
1,property_type,main residence,31,bigram,cleaned_seed
2,property_type,single story,25,bigram,cleaned_seed
3,property_type,duplex,17,unigram,sample_supported_seed
4,property_type,townhouse,13,unigram,sample_supported_seed
22,room,kitchen,958,unigram,ngram_added_seed
23,room,bedroom,770,unigram,ngram_added_seed
24,room,bathroom,543,unigram,ngram_added_seed
25,room,primary suite,339,bigram,cleaned_seed
26,room,living room,305,bigram,cleaned_seed


The taxonomy is broad enough for Week 1: it spans property type, rooms, amenities, interior features, exterior features, location, condition, and transaction language.

### 3.2 Taxonomy Coverage on Listing Remarks

Coverage is a practical check. This pass uses simple substring matching, which is good enough for a Week 1 baseline.

In [17]:
term_list = terms['term'].str.lower().tolist()

def matched_terms(text):
    text = text.lower()
    return [term for term in term_list if term in text]

listings['taxonomy_matches'] = listings['remarks'].apply(matched_terms)

pd.Series({
    'coverage': listings['taxonomy_matches'].str.len().gt(0).mean(),
    'avg_matches_per_listing': listings['taxonomy_matches'].str.len().mean(),
    'max_matches_in_listing': listings['taxonomy_matches'].str.len().max()
}).round(3)

coverage                    0.998
avg_matches_per_listing    20.322
max_matches_in_listing     53.000
dtype: float64

In [18]:
matched = Counter(term for row in listings['taxonomy_matches'] for term in row)

pd.DataFrame(matched.most_common(25), columns=['taxonomy_term', 'matched_listings'])

,taxonomy_term,matched_listings
0,spa,848
1,bedroom,801
2,kitchen,733
3,bathroom,562
4,den,550
5,garage,458
6,yard,449
7,pool,374
8,appliances,346
9,fireplace,337


---

## 4 Query

### 4.1 Query Set Overview

The sample queries define the kinds of user requests the later parser and classifier should handle.

In [19]:
queries_df['intent'].value_counts()

intent
amenity_search             12
property_search            10
price_filter               10
bed_bath_filter            10
location_search            10
interior_feature_search    10
exterior_feature_search    10
condition_search           10
property_type_search       10
investment_search          10
summary_request            10
open_house_search           8
Name: count, dtype: int64

In [20]:
pd.crosstab(queries_df['intent'], queries_df['difficulty'])

difficulty,hard,medium,simple
intent,,,
amenity_search,5,3,4
bed_bath_filter,4,3,3
condition_search,3,3,4
exterior_feature_search,3,3,4
interior_feature_search,3,3,4
investment_search,4,3,3
location_search,4,3,3
open_house_search,3,3,2
price_filter,4,3,3


In [21]:
queries_df.sample(8, random_state=7)[['id', 'query', 'intent', 'difficulty']]

,id,query,intent,difficulty
112,query_113,what are the main selling points,summary_request,simple
74,query_075,turnkey listings,condition_search,simple
79,query_080,"I do not want a project, only clean and ready ...",condition_search,hard
87,query_088,find townhomes with attached garage,property_type_search,medium
97,query_098,find family-friendly open houses this weekend ...,open_house_search,hard
13,query_014,condos in Irvine below 750000,price_filter,medium
101,query_102,properties with rental income,investment_search,simple
5,query_006,looking for a move-in ready place in Riverside...,property_search,medium


The query set covers basic filters, feature searches, open house requests, investment language, and summary requests. The mix of simple, medium, and hard examples should make later evaluation more realistic than a set of only clean keyword searches.

### 4.2 Taxonomy Coverage on Query Set

The listing coverage check shows whether the taxonomy appears in agent-written remarks. Query coverage asks a different question: does the same taxonomy also connect to buyer-written search language?

This does not need to be perfect. Some intents, such as price filters or summary requests, may rely more on numbers or task language than taxonomy terms.

In [22]:
queries_df['taxonomy_matches'] = queries_df['query'].apply(matched_terms)
queries_df['has_taxonomy_match'] = queries_df['taxonomy_matches'].str.len().gt(0)

pd.Series({
    'query_coverage': queries_df['has_taxonomy_match'].mean(),
    'avg_matches_per_query': queries_df['taxonomy_matches'].str.len().mean(),
    'max_matches_in_query': queries_df['taxonomy_matches'].str.len().max()
}).round(3)

query_coverage           0.700
avg_matches_per_query    1.117
max_matches_in_query     4.000
dtype: float64

In [23]:
queries_df.groupby('intent')['has_taxonomy_match'].mean().sort_values(ascending=False).round(3)

intent
amenity_search             0.917
exterior_feature_search    0.900
property_type_search       0.900
bed_bath_filter            0.800
condition_search           0.800
interior_feature_search    0.800
property_search            0.800
investment_search          0.600
location_search            0.600
price_filter               0.600
open_house_search          0.500
summary_request            0.100
Name: has_taxonomy_match, dtype: float64

In [24]:
queries_df.groupby('difficulty')['has_taxonomy_match'].mean().reindex(['simple', 'medium', 'hard']).round(3)

difficulty
simple    0.732
medium    0.784
hard      0.595
Name: has_taxonomy_match, dtype: float64